In [31]:
import pandas as pd 
import numpy as np 
import sklearn 
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from IPython.display import display

In [32]:
#!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

In [33]:
df = pd.read_csv("course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [34]:
#Checking if the missing values are presented in the features
df.isna().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


In [36]:
#classify variables based on dtypes
categorical_variables = df.select_dtypes(include="str").columns.to_list()
numerical_variables = ['annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted']

In [37]:
categorical_variables

['lead_source', 'industry', 'employment_status', 'location']

In [38]:
#filling missing values: 
#For categorical features, replace them with 'NA'
# For numerical features, replace with with 0.0

df[categorical_variables] = df[categorical_variables].fillna("NA")
df[numerical_variables] = df[numerical_variables].fillna(0)

In [39]:
df[df['location']== "NA"][['annual_income',"location"]].head()

,annual_income,location
3,0.0,NA
72,85628.0,NA
111,61509.0,NA
117,12000.0,NA
133,12000.0,NA


In [40]:
#the most frequent observation (mode) for the column industry
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [41]:
#the correlation matrix for the numerical features 
inter_lead_corr = df.interaction_count.corr(df.lead_score)
print(f"interaction_count and lead_score correlation {inter_lead_corr}")
print()
numco_lead_corr = df.number_of_courses_viewed.corr(df.lead_score)
print(f"number_of_courses_viewed and lead_score correlation {numco_lead_corr}")
print()
numco_inter_corr = df.number_of_courses_viewed.corr(df.interaction_count)
print(f"number_of_courses_viewed and interaction_count correlation {numco_inter_corr}")
print()
annual_inter_corr = df.annual_income.corr(df.interaction_count)
print(f"annual_income and interaction_count correlation {annual_inter_corr}")
print()

interaction_count and lead_score correlation 0.9157457980418677

number_of_courses_viewed and lead_score correlation 0.7572042392104887

number_of_courses_viewed and interaction_count correlation 0.7216090038937009

annual_income and interaction_count correlation 0.12284235135955954



In [42]:
df[["interaction_count","lead_score","number_of_courses_viewed","annual_income"]].corr()

,interaction_count,lead_score,number_of_courses_viewed,annual_income
interaction_count,1.000000,0.915746,0.721609,0.122842
lead_score,0.915746,1.000000,0.757204,0.229496
number_of_courses_viewed,0.721609,0.757204,1.000000,0.161300
annual_income,0.122842,0.229496,0.161300,1.000000


In [43]:
#Splitting the data
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [44]:
#glocal converted rate
global_rate = df_full_train.converted.mean()
print(global_rate)

0.57075


In [45]:
#calculating group converted rate
for cat in categorical_variables:
    mean_in = {}
    unique_industry = list(df_full_train[cat].value_counts().index)
    for c in unique_industry:
        mean_industry = float(df_full_train[df_full_train[cat]== c]["converted"].mean())
        mean_in[c] = round(mean_industry,2)
    print(cat)
    print(mean_in) 
    print()

lead_source
{'organic_search': 0.68, 'paid_ads': 0.43, 'social_media': 0.55, 'referral': 0.76, 'events': 0.41, 'NA': 0.61}

industry
{'technology': 0.61, 'retail': 0.55, 'healthcare': 0.56, 'finance': 0.61, 'education': 0.5, 'manufacturing': 0.55, 'NA': 0.59}

employment_status
{'employed': 0.66, 'student': 0.42, 'self_employed': 0.55, 'unemployed': 0.44, 'NA': 0.58}

location
{'north_america': 0.58, 'europe': 0.58, 'asia': 0.59, 'south_america': 0.53, 'africa': 0.55, 'NA': 0.6}



In [46]:
#feature important: converted rate and diff
for c in categorical_variables:
    df_mutality = df_full_train.groupby(c)["converted"].agg(["mean","count"])
    df_mutality["converted diff"] = df_mutality["mean"] - global_rate
    df_mutality["converted risk"] = df_mutality["mean"]/global_rate
    display(df_mutality)
    print()
    print()

,mean,count,converted diff,converted risk
lead_source,,,,
NA,0.610169,118,0.039419,1.069066
events,0.411290,496,-0.159460,0.720614
organic_search,0.677579,1008,0.106829,1.187174
paid_ads,0.427957,930,-0.142793,0.749815
referral,0.763844,614,0.193094,1.338316
social_media,0.547962,834,-0.022788,0.960073


,mean,count,converted diff,converted risk
industry,,,,
NA,0.587629,194,0.016879,1.029573
education,0.500000,518,-0.070750,0.876040
finance,0.613402,582,0.042652,1.074730
healthcare,0.559398,665,-0.011352,0.980111
manufacturing,0.548969,388,-0.021781,0.961838
retail,0.553741,735,-0.017009,0.970200
technology,0.611111,918,0.040361,1.070716


,mean,count,converted diff,converted risk
employment_status,,,,
NA,0.580247,162,0.009497,1.016639
employed,0.656459,2090,0.085709,1.150170
self_employed,0.548276,580,-0.022474,0.960623
student,0.419773,617,-0.150977,0.735476
unemployed,0.435572,551,-0.135178,0.763157


,mean,count,converted diff,converted risk
location,,,,
NA,0.603550,169,0.032800,1.057469
africa,0.546154,390,-0.024596,0.956906
asia,0.585308,844,0.014558,1.025507
europe,0.584821,896,0.014071,1.024654
north_america,0.576923,1092,0.006173,1.010816
south_america,0.525452,609,-0.045298,0.920633


In [47]:
###mutual information score between categorical features and converted --> a way to measure the importance of cat feature
def mi_information_score(feature):
    return mutual_info_score(feature,df_full_train.converted)

In [48]:
categorical_mutual_info = df_full_train[categorical_variables].apply(mi_information_score).sort_values(ascending=False)
print(round(categorical_mutual_info,2))

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64
